# Capítulo 8 — Qué hace realmente un ordenador cuando resuelve una ecuación

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## De la EDO a la EDP: ¿por qué hay un paso de tiempo máximo?

Ecuación del calor resuelta con diferencias finitas explícitas, justo por
debajo y justo por encima del límite CFL.

La figura responde: ¿qué relación hay entre el paso espacial y el temporal, y
qué pasa si la violas?

Ejecutar:  python fig_cfl_calor.py

*(script original: `codigo/fig_cfl_calor.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

NX, L, D = 101, 1.0, 1.0
dx = L / (NX - 1)
x = np.linspace(0, L, NX)
u0 = np.where(np.abs(x - 0.5) < 0.12, 1.0, 0.0)


def resuelve(r, t_final):
    """r = D dt / dx^2  es el número adimensional que decide todo."""
    dt = r * dx**2 / D
    n = int(t_final / dt)
    u = u0.copy()
    guardados = []
    for i in range(n):
        u[1:-1] = u[1:-1] + r * (u[2:] - 2 * u[1:-1] + u[:-2])
        if i in {0, n // 8, n // 3, n - 1}:
            guardados.append(u.copy())
    return guardados, dt, n


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.2, 4.1))

for ax, r, titulo in [(ax1, 0.49, "$r = 0{,}49$: estable"),
                      (ax2, 0.51, "$r = 0{,}51$: inestable")]:
    guardados, dt, n = resuelve(r, 0.004)
    for k, u in enumerate(guardados):
        ax.plot(x, u, lw=1.6, alpha=0.85,
                label=f"paso {[0, n//8, n//3, n-1][k]}")
    ax.set_xlabel("$x$"), ax.set_ylabel("$u$")
    ax.set_title(f"{titulo}   ($\\Delta t = {dt:.1e}$)", fontsize=10)
    ax.legend(fontsize=7.6)
    print(f"r={r}: máximo final = {abs(guardados[-1]).max():.3e}")

ax2.set_ylim(-3, 3)
ax2.text(0.02, 2.2, "la solución oscila\ny crece sin control", fontsize=8.6,
         color=C.red)
ax1.text(0.02, 0.85, "difunde, como debe", fontsize=8.6, color=C.green)

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Cuánto error llevas puesto antes de empezar a calcular?

Tres caras de la aritmética de coma flotante: el espaciado de los números
representables, la cancelación catastrófica y su remedio algebraico.

La figura responde: ¿por qué dos fórmulas matemáticamente idénticas dan
resultados distintos, y cuál hay que usar?

Ejecutar:  python fig_coma_flotante.py

*(script original: `codigo/fig_coma_flotante.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()
fig, axes = plt.subplots(1, 3, figsize=(11.4, 3.9))

# --- 1. Espaciado de los flotantes ---------------------------------------
ax = axes[0]
x = np.logspace(-8, 8, 200)
ax.loglog(x, np.spacing(x), color=C.blue, lw=2)
ax.axhline(np.spacing(1.0), color=C.grey, ls="--", lw=1.1)
ax.text(1e-7, np.spacing(1.0) * 1.6, r"$\epsilon_{\mathrm{maq}}=2{,}2\times10^{-16}$",
        fontsize=8.4, color=C.grey)
ax.set_xlabel("valor $x$")
ax.set_ylabel("distancia al siguiente double")
ax.set_title("La resolución depende del tamaño", fontsize=10)
ax.annotate("cerca de $10^8$ el hueco\nya es $10^{-8}$", xy=(1e8, np.spacing(1e8)),
            xytext=(1e-6, 1e-10), fontsize=8.2, color=C.ink,
            arrowprops=dict(arrowstyle="->", color=C.ink, lw=1.0))

# --- 2. Cancelación catastrófica -----------------------------------------
ax = axes[1]
h = np.logspace(-12, 0, 300)
ingenua = (1 - np.cos(h)) / h**2
estable = 2 * (np.sin(h / 2) / h) ** 2
ax.semilogx(h, ingenua, color=C.red, lw=1.6, label=r"$(1-\cos h)/h^2$")
ax.semilogx(h, estable, color=C.blue, lw=2.0,
            label=r"$2\,[\sin(h/2)/h]^2$")
ax.axhline(0.5, color=C.ink, ls="--", lw=1.1)
ax.text(2e-12, 0.53, "valor exacto: 1/2", fontsize=8.4, color=C.ink)
ax.set_ylim(-0.15, 0.75)
ax.set_xlabel("$h$"), ax.set_ylabel("valor calculado")
ax.set_title("Dos fórmulas idénticas en el papel", fontsize=10)
ax.legend(fontsize=8, loc="lower right")

# --- 3. Derivada numérica: el compromiso ---------------------------------
ax = axes[2]
h = np.logspace(-16, 0, 400)
x0 = 1.0
adelante = np.abs((np.sin(x0 + h) - np.sin(x0)) / h - np.cos(x0))
centrada = np.abs((np.sin(x0 + h) - np.sin(x0 - h)) / (2 * h) - np.cos(x0))
ax.loglog(h, np.maximum(adelante, 1e-18), color=C.red, lw=1.6,
          label="hacia delante, $O(h)$")
ax.loglog(h, np.maximum(centrada, 1e-18), color=C.blue, lw=1.8,
          label="centrada, $O(h^2)$")
ax.loglog(h, h / 2, ":", color=C.grey, lw=1.2)
ax.loglog(h, 2.2e-16 / h, ":", color=C.grey, lw=1.2)
ax.text(8e-1, 2e-12, "manda el error\nde truncamiento", fontsize=7.8,
        color=C.grey, ha="right", va="bottom")
ax.text(3e-16, 2e-12, "manda el error\nde redondeo", fontsize=7.8,
        color=C.grey, ha="left", va="bottom")
ax.set_xlabel("paso $h$"), ax.set_ylabel("error absoluto")
ax.set_title("Ni muy grande ni muy pequeño", fontsize=10)
ax.legend(fontsize=8, loc="upper center")
ax.set_ylim(1e-13, 1e1)

print(f"0.1 + 0.2 == 0.3 ?  {0.1 + 0.2 == 0.3}")
print(f"0.1 + 0.2 = {0.1 + 0.2:.20f}")
print(f"h óptimo centrada ≈ {(2.2e-16)**(1/3):.2e}, "
      f"error mínimo ≈ {centrada.min():.2e}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Qué pasa si integras una órbita durante mucho tiempo?

Oscilador armónico integrado con Euler explícito, Euler implícito, RK4 y Euler
simpléctico. Se dibuja la energía y el retrato de fases.

La figura responde: ¿por qué un método de orden 1 puede batir a uno de orden 4
en el largo plazo?

Ejecutar:  python fig_energia_integradores.py

*(script original: `codigo/fig_energia_integradores.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

H, N = 0.05, 40_000          # paso y número de pasos
t = np.arange(N + 1) * H


def integra(metodo):
    q = np.empty(N + 1)
    p = np.empty(N + 1)
    q[0], p[0] = 1.0, 0.0
    for i in range(N):
        if metodo == "euler":
            q[i + 1] = q[i] + H * p[i]
            p[i + 1] = p[i] - H * q[i]
        elif metodo == "implicito":
            den = 1 + H**2
            q[i + 1] = (q[i] + H * p[i]) / den
            p[i + 1] = (p[i] - H * q[i]) / den
        elif metodo == "simplectico":
            p[i + 1] = p[i] - H * q[i]           # primero p, con q antiguo
            q[i + 1] = q[i] + H * p[i + 1]       # después q, con p nuevo
        elif metodo == "rk4":
            def der(y):
                return np.array([y[1], -y[0]])
            y = np.array([q[i], p[i]])
            k1 = der(y); k2 = der(y + H * k1 / 2)
            k3 = der(y + H * k2 / 2); k4 = der(y + H * k3)
            y = y + H * (k1 + 2 * k2 + 2 * k3 + k4) / 6
            q[i + 1], p[i + 1] = y
    return q, p


METODOS = [("euler", "Euler explícito (orden 1)", C.red),
           ("implicito", "Euler implícito (orden 1)", C.ochre),
           ("rk4", "Runge–Kutta 4 (orden 4)", C.blue),
           ("simplectico", "Euler simpléctico (orden 1)", C.green)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

for clave, nombre, color in METODOS:
    q, p = integra(clave)
    E = 0.5 * (q**2 + p**2)
    ax1.plot(t, E, color=color, lw=1.5, label=nombre)
    ax2.plot(q[-2000:], p[-2000:], color=color, lw=1.0, alpha=0.85)
    print(f"{nombre:32s} E final = {E[-1]:.4f}  (exacta 0.5)")

ax1.set_yscale("log")
ax1.axhline(0.5, color=C.ink, ls="--", lw=1.2)
ax1.text(50, 0.55, "energía exacta", fontsize=8.4, color=C.ink)
ax1.set_xlabel("tiempo"), ax1.set_ylabel("energía")
ax1.set_title("2000 periodos de un oscilador armónico")
ax1.legend(fontsize=7.8, loc="center left")
ax1.set_ylim(1e-3, 1e3)

ax2.set_xlabel("$q$"), ax2.set_ylabel("$p$")
ax2.set_title("Últimos 2000 pasos en el plano de fases")
ax2.set_aspect("equal")
ax2.set_xlim(-2.2, 2.2), ax2.set_ylim(-2.2, 2.2)
circulo = np.linspace(0, 2 * np.pi, 200)
ax2.plot(np.cos(circulo), np.sin(circulo), "--", color=C.ink, lw=1.2)

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Por qué un paso «pequeño» a veces explota?

Regiones de estabilidad absoluta en el plano complejo y un ejemplo rígido
donde Euler explícito revienta y el implícito no se inmuta.

La figura responde: ¿qué limita el paso, la precisión que quiero o la
estabilidad del método?

Ejecutar:  python fig_estabilidad.py

*(script original: `codigo/fig_estabilidad.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.3))

# --- Regiones de estabilidad ---------------------------------------------
re = np.linspace(-3.5, 1.5, 700)
im = np.linspace(-3.5, 3.5, 700)
Z = re[None, :] + 1j * im[:, None]

R_euler = 1 + Z
R_heun = 1 + Z + Z**2 / 2
R_rk4 = 1 + Z + Z**2 / 2 + Z**3 / 6 + Z**4 / 24

for R, nombre, color in [(R_euler, "Euler explícito", C.red),
                         (R_heun, "Heun (RK2)", C.ochre),
                         (R_rk4, "RK4", C.blue)]:
    ax1.contour(re, im, np.abs(R), levels=[1.0], colors=[color], linewidths=2)
    ax1.contourf(re, im, np.abs(R), levels=[0, 1.0], colors=[color], alpha=0.13)
    ax1.plot([], [], color=color, lw=2, label=nombre)

ax1.axhline(0, color=C.ink, lw=0.8), ax1.axvline(0, color=C.ink, lw=0.8)
ax1.set_xlabel(r"$\mathrm{Re}(h\lambda)$")
ax1.set_ylabel(r"$\mathrm{Im}(h\lambda)$")
ax1.set_title("Regiones de estabilidad absoluta")
ax1.legend(fontsize=8, loc="upper left")
ax1.set_aspect("equal")
ax1.text(-2.6, -3.1, "el implícito es estable\nen TODO el semiplano izquierdo",
         fontsize=8.2, color=C.green)

# --- Un problema rígido ---------------------------------------------------
LAMBDA = -1000.0        # modo rápido
T = 0.05


def exacta(t):
    return np.exp(LAMBDA * t)


for h, color, estilo in [(0.0018, C.red, "-"), (0.0022, C.ink, "-")]:
    n = int(T / h)
    t = np.arange(n + 1) * h
    y = np.empty(n + 1)
    y[0] = 1.0
    for i in range(n):
        y[i + 1] = y[i] + h * LAMBDA * y[i]
    ax2.plot(t, y, estilo, color=color, lw=1.6, marker="o", ms=3,
             label=f"Euler explícito, $h\\lambda$ = {h*LAMBDA:.1f}")

# Euler implícito con el paso grande
h = 0.0022
n = int(T / h)
t = np.arange(n + 1) * h
y = np.empty(n + 1)
y[0] = 1.0
for i in range(n):
    y[i + 1] = y[i] / (1 - h * LAMBDA)
ax2.plot(t, y, color=C.green, lw=2.0, marker="s", ms=3,
         label=f"Euler implícito, $h\\lambda$ = {h*LAMBDA:.1f}")

tt = np.linspace(0, T, 400)
ax2.plot(tt, exacta(tt), "--", color=C.grey, lw=1.4, label="exacta")
ax2.set_yscale("symlog", linthresh=1e-3)
ax2.set_xlabel("$t$"), ax2.set_ylabel("$y$")
ax2.set_title(r"$\dot y = -1000\,y$: el paso lo fija la estabilidad")
ax2.legend(fontsize=7.6, loc="lower left")

print(f"límite de estabilidad de Euler explícito: h < 2/|lambda| = "
      f"{2/abs(LAMBDA):.4f}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Cómo se comprueba que un método numérico hace lo que promete?

Error global frente al paso, en ejes logarítmicos, para Euler, Euler mejorado
y Runge-Kutta 4, sobre un problema con solución exacta.

La figura responde: ¿qué es el «orden» de un método, y cómo se mide en dos
líneas?

Ejecutar:  python fig_orden_convergencia.py

*(script original: `codigo/fig_orden_convergencia.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

# Problema de prueba: y' = -2y + sin(t),  y(0)=1,  con solución exacta
def f(t, y):
    return -2 * y + np.sin(t)


def exacta(t):
    # solución particular + homogénea, ajustada a y(0)=1
    a = (2 * np.sin(t) - np.cos(t)) / 5
    return (1 + 1 / 5) * np.exp(-2 * t) + a


def euler(f, y0, t0, t1, n):
    h = (t1 - t0) / n
    y, t = y0, t0
    for _ in range(n):
        y += h * f(t, y)
        t += h
    return y


def heun(f, y0, t0, t1, n):
    h = (t1 - t0) / n
    y, t = y0, t0
    for _ in range(n):
        k1 = f(t, y)
        k2 = f(t + h, y + h * k1)
        y += h * (k1 + k2) / 2
        t += h
    return y


def rk4(f, y0, t0, t1, n):
    h = (t1 - t0) / n
    y, t = y0, t0
    for _ in range(n):
        k1 = f(t, y)
        k2 = f(t + h / 2, y + h * k1 / 2)
        k3 = f(t + h / 2, y + h * k2 / 2)
        k4 = f(t + h, y + h * k3)
        y += h * (k1 + 2 * k2 + 2 * k3 + k4) / 6
        t += h
    return y


T = 2.0
ns = np.array([2**k for k in range(2, 20)])
hs = T / ns
y_ref = exacta(T)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.2, 4.2))

for metodo, nombre, color, orden in [(euler, "Euler", C.red, 1),
                                     (heun, "Euler mejorado (Heun)", C.ochre, 2),
                                     (rk4, "Runge–Kutta 4", C.blue, 4)]:
    err = np.array([abs(metodo(f, 1.0, 0.0, T, int(n)) - y_ref) for n in ns])
    err = np.maximum(err, 1e-17)
    ax1.loglog(hs, err, "o-", color=color, ms=4, lw=1.5, label=nombre)
    # pendiente medida en la zona limpia
    m = (err > 1e-13) & (err < 1e-3)   # sólo el régimen asintótico limpio
    if m.sum() > 2:
        p = np.polyfit(np.log10(hs[m]), np.log10(err[m]), 1)[0]
        print(f"{nombre:24s} orden medido = {p:.2f}  (teórico {orden})")

for orden, color in [(1, C.red), (2, C.ochre), (4, C.blue)]:
    ax1.loglog(hs, 0.3 * hs**orden, ":", color=color, lw=1.0)
ax1.set_xlabel("paso $h$"), ax1.set_ylabel("error global en $t=2$")
ax1.set_title("La pendiente en log-log **es** el orden")
ax1.legend(fontsize=8, loc="lower right")
ax1.set_ylim(1e-17, 1e0)

# --- Panel 2: coste, no paso ---------------------------------------------
for metodo, nombre, color, evals in [(euler, "Euler", C.red, 1),
                                     (heun, "Heun", C.ochre, 2),
                                     (rk4, "RK4", C.blue, 4)]:
    err = np.array([abs(metodo(f, 1.0, 0.0, T, int(n)) - y_ref) for n in ns])
    ax2.loglog(ns * evals, np.maximum(err, 1e-17), "o-", color=color, ms=4,
               lw=1.5, label=nombre)
ax2.set_xlabel("evaluaciones de $f$ (coste real)")
ax2.set_ylabel("error global")
ax2.set_title("Lo que importa no es el paso: es el coste")
ax2.legend(fontsize=8)
ax2.set_ylim(1e-17, 1e0)
ax2.annotate("para el mismo coste,\nRK4 acierta $10^{9}$ veces mejor",
             xy=(1e3, 1e-13), xytext=(1.2e1, 1e-9), fontsize=8.4, color=C.blue,
             arrowprops=dict(arrowstyle="->", color=C.blue, lw=1.0))

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
